In [1]:
import io
import boto3
import pandas as pd
import polars as pl

# Connexion S3
s3_client = boto3.client(
    "s3",
    endpoint_url="http://192.168.1.230:30137",
    aws_access_key_id="datalab-team",
    aws_secret_access_key="minio-datalabteam123",
    verify=False
)

# Paramètres
bucket_name = "staging"
object_key = "panel_admin/references/anstat_2025.txt.xlsx"

# Télécharger le fichier dans un buffer
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
buf = io.BytesIO(obj['Body'].read())  # <-- Lire le contenu dans le buffer
buf.seek(0)

# Charger avec pandas
df_ref = pd.read_excel(buf)

print(df_ref.head())

          SECTEUR CATEGORIE GRADE  INDICE ECHELON/CLASSE  Salaire de Base  \
0  Barême Général         A    A7    2350             21        548621.60   
1  Barême Général         A    A7    2295             33        535781.52   
2  Barême Général         A    A7    2255             32        526443.28   
3  Barême Général         A    A7    2170             31        506599.52   
4  Barême Général         A    A7    2135             44        498428.56   

   Résidence  Logement  Transport  Allocations\n (2 enfants)  \
0  82293.240     90000      15000                      15000   
1  80367.228     90000      15000                      15000   
2  78966.492     90000      15000                      15000   
3  75989.928     90000      15000                      15000   
4  74764.284     90000      15000                      15000   

   AUTRES INDEMNITES  Montant brut  Charge patronale          ITS  \
0                NaN    750914.840      91455.220720  91992.11640   
1             

In [6]:
print(df_ref.columns)

Index(['SECTEUR', 'CATEGORIE', 'GRADE', 'INDICE', 'ECHELON/CLASSE',
       'Salaire de Base', 'Résidence', 'Logement', 'Transport',
       'Allocations\n (2 enfants)', 'AUTRES INDEMNITES', 'Montant brut',
       'Charge patronale', 'ITS', 'PENSION', 'MUGEF-CI/CMU', 'NET',
       'BASE IMPOSABLE'],
      dtype='object')


In [7]:
# Calculer la moyenne, min et max par SECTEUR et GRADE
stats_secteur_grade = df_ref.groupby(['SECTEUR', 'GRADE'])[['Montant brut', 'NET']].agg(['mean', 'min', 'max']).reset_index()

# Aplatir les colonnes multi-index pour plus de clarté
stats_secteur_grade.columns = ['SECTEUR', 'GRADE', 
                               'moy_brut', 'min_brut', 'max_brut',
                               'moy_net', 'min_net', 'max_net']

print("Statistiques par SECTEUR et GRADE :")
print(stats_secteur_grade.head())

Statistiques par SECTEUR et GRADE :
          SECTEUR GRADE       moy_brut    min_brut    max_brut        moy_net  \
0  Barême Général    A3  434989.604000  312833.752  570569.176  371170.698469   
1  Barême Général    A4  498891.002769  340284.588  662453.868  419815.275825   
2  Barême Général    A5  534379.910462  379077.796  687823.356  450859.491210   
3  Barême Général    A6  589107.384308  473043.836  695877.588  490111.142201   
4  Barême Général    A7  699569.111000  646209.824  750914.840  569374.635128   

         min_net        max_net  
0  283726.382896  468335.725248  
1  306651.569024  537093.332064  
2  339646.663808  560946.289888  
3  406827.976928  566725.726624  
4  531085.866752  606218.544320  


In [8]:
# Nom du bucket et chemin de destination
bucket_name = "staging"
object_key = "panel_admin/exports_gold/stats_secteur_grade_Solde.xlsx"

# Créer un buffer pour écrire le fichier Excel en mémoire
excel_buffer = io.BytesIO()

# Exporter le DataFrame vers le buffer
stats_secteur_grade.to_excel(excel_buffer, index=False)

# Repositionner le curseur au début
excel_buffer.seek(0)

# Télécharger le fichier vers S3
s3_client = boto3.client(
    "s3",
    endpoint_url="http://192.168.1.230:30137",
    aws_access_key_id="datalab-team",
    aws_secret_access_key="minio-datalabteam123",
    verify=False
)

s3_client.put_object(Bucket=bucket_name, Key=object_key, Body=excel_buffer.getvalue())

print(f"Fichier exporté avec succès vers s3://{bucket_name}/{object_key}")

Fichier exporté avec succès vers s3://staging/panel_admin/exports_gold/stats_secteur_grade_Solde.xlsx


In [9]:
# --- Chemin du fichier ---
bucket_name = "staging"
object_key  = "panel_admin/exports_gold/stats_secteur_grade_Solde.xlsx"

# --- Télécharger le fichier en mémoire ---
excel_buffer = io.BytesIO()
s3_client.download_fileobj(bucket_name, object_key, excel_buffer)

# Repositionner le curseur au début
excel_buffer.seek(0)

# --- Lire avec pandas ---
df = pd.read_excel(excel_buffer)

# Vérification
print(df.head())
print(df.columns)

          SECTEUR GRADE       moy_brut    min_brut    max_brut        moy_net  \
0  Barême Général    A3  434989.604000  312833.752  570569.176  371170.698469   
1  Barême Général    A4  498891.002769  340284.588  662453.868  419815.275825   
2  Barême Général    A5  534379.910462  379077.796  687823.356  450859.491210   
3  Barême Général    A6  589107.384308  473043.836  695877.588  490111.142201   
4  Barême Général    A7  699569.111000  646209.824  750914.840  569374.635128   

         min_net        max_net  
0  283726.382896  468335.725248  
1  306651.569024  537093.332064  
2  339646.663808  560946.289888  
3  406827.976928  566725.726624  
4  531085.866752  606218.544320  
Index(['SECTEUR', 'GRADE', 'moy_brut', 'min_brut', 'max_brut', 'moy_net',
       'min_net', 'max_net'],
      dtype='object')


In [10]:
# =========================================
# 1. Jointure avec table de référence pour max brut
# =========================================
# Sélection des colonnes dans pandas
table_max_brut = df[["SECTEUR", "GRADE", "max_brut"]].copy()

# Renommage des colonnes pour faciliter la jointure
table_max_brut = table_max_brut.rename(columns={"GRADE": "GRADE_REF", "SECTEUR": "BAREME_REF"})
table_max_brut

,BAREME_REF,GRADE_REF,max_brut
0,Barême Général,A3,570569.176
1,Barême Général,A4,662453.868
2,Barême Général,A5,687823.356
3,Barême Général,A6,695877.588
4,Barême Général,A7,750914.840
...,...,...,...
60,para militaire,A7,970914.840
61,para militaire,B3,605728.580
62,para militaire,C1,391823.616
63,para militaire,C2,406589.708


In [ ]:
# ============================================================
# PANEL ADMIN — ÉTAPE 7 : CALCUL DES INDICATEURS SALAIRES
# ============================================================
# Produit les indicateurs statistiques complets sur les salaires
# à partir des fichiers Gold panel.
#
# Prérequis : 03b_imputer_grades_silver.py et 06_compiler_panel.py
# doivent avoir été exécutés. Les colonnes GRADE et GRADE_SOURCE
# sont lues directement depuis le Gold panel — aucune imputation
# n'est recalculée ici.
#
# Paramètres configurables :
#   MODE_SALAIRE_BRUT : True  = agrégation par cle_unique (salaire brut)
#                       False = agrégation par matricule (revenu salarial)
#   PERIODE_RECENTE   : True  = années récentes uniquement (2024-2025)
#                       False = période complète (2015 → dernière année)
#   INCLURE_ZEROS     : True  = inclure salaires = 0 ou NA
#                       False = salaire strictement > 0
#
# Tables produites (Excel multi-feuilles) :
#   REVENU_Grade_Detail      : Effectif, moy, méd, min, max, P25, P75 par grade
#   REVENU_Grade_Sexe        : Idem, par grade × sexe
#   REVENU_CITP_Detail       : Idem, par CITP
#   SALAIRE_BRUT_Moyen       : Salaire brut moyen par année × mois × grade (pivot)
#   SALAIRE_BRUT_Median      : Salaire brut médian par année × mois × grade (pivot)
#   MULTI_Par_Grade_NbPostes : Effectifs par grade × nombre de postes (pivot)
#   MULTI_Distribution       : Distribution globale mono/multi-postes
#   MODELE_ANSTAT_2024       : Modèle statistique ANSTAT — année 2024
#   MODELE_ANSTAT_2025       : Modèle statistique ANSTAT — année 2025
#
# Source  : s3://gold/panel_admin/panel_YYYY.parquet
# Sortie  : s3://staging/panel_admin/exports_gold/indicateurs_*.xlsx
#
# Dépendances :
#   pip install polars boto3 python-dotenv openpyxl
# ============================================================

import gc
import io
import os
import re
import tempfile
import unicodedata
from dotenv import load_dotenv
import boto3
from botocore.client import Config
import polars as pl
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

load_dotenv(".env")

# ============================================================
# PARAMÈTRES DE CONFIGURATION
# ============================================================

# True = Salaire brut (cle_unique) | False = Revenu salarial (matricule)
MODE_SALAIRE_BRUT = True

# True = 2024-2025 | False = 2015 → dernière année disponible
PERIODE_RECENTE   = True

# True = inclure zéros | False = salaire strictement > 0
INCLURE_ZEROS     = True

# Seuil de winsorisation (ex: 0.01 = écrêtage à P1/P99)
# Mettre à None pour désactiver
SEUIL_WINS        = 0.01

# Effectif minimum par cellule mois × grade × CITP pour appliquer la
# winsorisation CITP. En dessous, repli sur mois × grade uniquement.
MIN_OBS_WINS      = 100

# ============================================================

# LORSQU'ON TRAVAILLE DEPUIS SA MACHINE LOCAL
MINIO_ENDPOINT   = os.getenv("MINIO_ENDPOINT",   "http://192.168.1.230:30137")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "datalab-team")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minio-datalabteam123")

BUCKET_GOLD    = os.getenv("BUCKET_GOLD",    "gold")
BUCKET_STAGING = os.getenv("BUCKET_STAGING", "staging")
PREFIX_PANEL   = "panel_admin"
PREFIX_EXPORTS = "panel_admin/exports_gold"
KEY_CORRESPONDANCE = "panel_admin/references/CORRESPONDANCE_GRILLE_EMPLOI.xlsx"

_mode_label    = "salaire_brut"  if MODE_SALAIRE_BRUT else "revenu_salarial"
_periode_label = "2024_2025"     if PERIODE_RECENTE   else "2015_recent"
_zeros_label   = "avec_zeros"    if INCLURE_ZEROS     else "sans_zeros"
OUTPUT_KEY     = (f"{PREFIX_EXPORTS}/indicateurs_{_mode_label}"
                  f"_{_periode_label}_{_zeros_label}.xlsx")

# Libellés mois français
MOIS_FR = {
    1: "Janvier",  2: "Février",   3: "Mars",      4: "Avril",
    5: "Mai",      6: "Juin",      7: "Juillet",   8: "Août",
    9: "Septembre",10: "Octobre",  11: "Novembre", 12: "Décembre",
}

GRADES_VALIDES = (
    [f"A{i}" for i in range(1, 8)] +
    [f"B{i}" for i in range(1, 7)] +
    [f"C{i}" for i in range(1, 6)] +
    [f"D{i}" for i in range(1, 4)]
)

# Composantes du salaire brut (codes F2, suffixe _f2 dans le Gold)
COMPOSANTES_SALAIRE = ["124_f2", "228_f2", "271_f2", "421_f2", "425_f2"]

# --- CLIENT S3 ---
s3 = boto3.client(
    "s3",
    endpoint_url          = MINIO_ENDPOINT,
    aws_access_key_id     = MINIO_ACCESS_KEY,
    aws_secret_access_key = MINIO_SECRET_KEY,
    config                = Config(
        signature_version = "s3v4",
        retries           = {"max_attempts": 5, "mode": "adaptive"},
        connect_timeout   = 30,
        read_timeout      = 120,
    ),
    region_name = "us-east-1",
    verify      = False,
)

print("=" * 70)
print(f"INDICATEURS SALAIRES — Mode : "
      f"{'Salaire brut (cle_unique)' if MODE_SALAIRE_BRUT else 'Revenu salarial (matricule)'}"
      f" | Période : {'2024-2025' if PERIODE_RECENTE else '2015 → récent'}")
print("=" * 70)
print()

# ============================================================
# FONCTIONS UTILITAIRES
# ============================================================

def lire_parquet_s3(bucket: str, key: str) -> pl.DataFrame:
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    return pl.read_parquet(buf)


# ============================================================
# CONSTANTES BARÈME
# ============================================================

# Pour ENS SEC & PRIM : grades B/C/D = Primaire, grades A = Secondaire
GRADES_PRIMAIRE = {"B3", "C3", "D1", "D2"}

# Ordre d'affichage dans la feuille MODELE_ANSTAT
ORDRE_MODELE_ANSTAT = [
    ("TOUS SECTEURS CONFONDUS",                       "_GLOBAL_"),
    ("SECTEUR EDUCATION",                             None),
    ("Primaire",                                      "ENS SEC & PRIM_PRIM"),
    ("Secondaire",                                    "ENS SEC & PRIM_SEC"),
    ("Supérieur",                                     "ENS SUP"),
    ("SECTEUR SANTE",                                 None),
    ("Cadres supérieurs",                             "Cadre-Sup-Santé Generaliste & Specialiste"),
    ("Personnels Techniques",                         "Tech-Santé"),
    ("AUTRES SECTEURS",                               None),
    ("Para-militaire / Douanes / Eaux & Forêts",      "police"),
    ("Barème Général",                                "Barême Général"),
    ("PERSONNELS SOUS STATUT PARTICULIER",            None),
    ("Magistrat",                                     "MAGISTRAT "),
    ("Corps préfectoral",                             "CORPS PREF"),
    ("Corps diplomatique",                            "CORPS DIPL"),
    ("Greffier",                                      "GREFFE"),
]


def normaliser_pour_matching(texte) -> str | None:
    """Normalisation textuelle pour fuzzy join — identique à l'étape 03."""
    if texte is None:
        return None
    texte = str(texte).strip()
    if not texte:
        return None
    texte = re.sub(r"[-]", " ", texte)
    texte = re.sub(r"[.,;:/()\\[\]]", " ", texte)
    texte = re.sub(r"[''\u2019]", " ", texte)
    texte = texte.upper()
    texte = unicodedata.normalize("NFKD", texte)
    texte = "".join(c for c in texte if not unicodedata.combining(c))
    texte = re.sub(r"\s+", " ", texte).strip()
    return texte if texte else None


def charger_table_bareme() -> pl.DataFrame | None:
    """
    Charge la table CORRESPONDANCE_GRILLE_EMPLOI depuis staging et
    renvoie une table de jointure (emploi_norm → bareme, grade_ref).
    Renvoie None si le fichier est absent (étape non bloquante).
    """
    with tempfile.NamedTemporaryFile(suffix=".xlsx", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        s3.download_file(BUCKET_STAGING, KEY_CORRESPONDANCE, tmp_path)
        df = pl.read_excel(tmp_path)
        df = df.rename({
            "BAREME / GRILLE SALARIALE": "bareme",
            "GRADE": "grade_ref",
            "EMPLOI": "emploi_ref",
        })
        df = df.with_columns(
            pl.col("emploi_ref")
            .map_elements(normaliser_pour_matching, return_dtype=pl.Utf8)
            .alias("emploi_ref_norm")
        )
        table = (
            df.filter(pl.col("emploi_ref_norm").is_not_null())
            .select(["emploi_ref_norm", "bareme", "grade_ref"])
            .unique(subset=["emploi_ref_norm"])
        )
        print(f"  ✓ Table barème chargée : {len(table)} emplois")
        return table
    except Exception as e:
        print(f"  ⚠️  Table barème non disponible ({e}) — feuilles barème ignorées")
        return None
    finally:
        os.unlink(tmp_path)


def fuzzy_join_bareme(df: pl.DataFrame, table_join: pl.DataFrame,
                      col_emploi: str) -> pl.DataFrame:
    """
    Ajoute les colonnes 'bareme' et 'grade_ref' au DataFrame
    via fuzzy join sur le libellé emploi normalisé.
    """
    df = df.with_columns(
        pl.col(col_emploi)
        .map_elements(normaliser_pour_matching, return_dtype=pl.Utf8)
        .alias("_emploi_norm")
    )
    df = df.join(table_join, left_on="_emploi_norm", right_on="emploi_ref_norm",
                 how="left").drop("_emploi_norm")
    nb_match  = int(df["bareme"].is_not_null().sum())
    taux      = 100 * nb_match / len(df)
    print(f"  ✓ Fuzzy join barème : {nb_match:,} / {len(df):,} ({taux:.1f}%)")
    if taux < 20:
        print("  ⚠️  Matching faible — vérifiez CORRESPONDANCE_GRILLE_EMPLOI.xlsx")
    return df


def ajouter_bareme_detail(df: pl.DataFrame, col_grade: str) -> pl.DataFrame:
    """Scinde ENS SEC & PRIM en Primaire (B/C/D) et Secondaire (A)."""
    return df.with_columns(
        pl.when(
            (pl.col("bareme") == "ENS SEC & PRIM")
            & pl.col(col_grade).is_in(GRADES_PRIMAIRE)
        ).then(pl.lit("ENS SEC & PRIM_PRIM"))
        .when(
            (pl.col("bareme") == "ENS SEC & PRIM")
            & ~pl.col(col_grade).is_in(GRADES_PRIMAIRE)
        ).then(pl.lit("ENS SEC & PRIM_SEC"))
        .otherwise(pl.col("bareme"))
        .alias("bareme_detail")
    )


def grade_order_value(grade: str) -> int:
    """Valeur ordinale d'un grade pour sélectionner le grade principal."""
    if re.match(r"^[ABCD][0-9]$", grade):
        base = {"A": 80, "B": 70, "C": 60, "D": 50}[grade[0]]
        return base + int(grade[1])
    return 0


def compute_indicators(df: pl.DataFrame, group_cols: list[str],
                       value_col: str = "REVENU_TOTAL") -> pl.DataFrame:
    """Calcule les indicateurs statistiques complets par groupe."""
    return (
        df.group_by(group_cols)
        .agg([
            pl.len().alias("Effectif"),
            pl.col(value_col).mean().round(0).alias("Revenu_moyen"),
            pl.col(value_col).median().round(0).alias("Revenu_median"),
            pl.col(value_col).min().round(0).alias("Revenu_min"),
            pl.col(value_col).max().round(0).alias("Revenu_max"),
            pl.col(value_col).quantile(0.25, interpolation="nearest")
              .round(0).alias("Revenu_p25"),
            pl.col(value_col).quantile(0.75, interpolation="nearest")
              .round(0).alias("Revenu_p75"),
        ])
        .sort(group_cols)
    )


def construire_feuille_anstat(
    wb: Workbook,
    sheet_name: str,
    annee_label: str,
    global_brut: dict,
    stats_bareme_detail: pl.DataFrame,
) -> None:
    """
    Crée une feuille MODELE_ANSTAT pour une année donnée.
    Appelée en boucle pour 2024 et 2025.
    """
    ws = wb.create_sheet(sheet_name)
    ws.sheet_view.showGridLines = False

    fill_titre   = PatternFill("solid", fgColor="1F4E79")
    fill_entete  = PatternFill("solid", fgColor="2E75B6")
    fill_global  = PatternFill("solid", fgColor="FFF2CC")
    fill_secteur = PatternFill("solid", fgColor="D6E4F0")
    fill_alt     = [PatternFill("solid", fgColor="EBF3FB"),
                    PatternFill("solid", fgColor="FFFFFF")]

    font_titre   = Font(bold=True, color="FFFFFF", size=12)
    font_entete  = Font(bold=True, color="FFFFFF", size=10)
    font_global  = Font(bold=True, color="7F6000", size=11)
    font_secteur = Font(bold=True, color="1F4E79", size=10)
    font_sous    = Font(size=10, italic=True)
    font_montant = Font(size=10)

    al_centre = Alignment(horizontal="center", vertical="center")
    al_gauche = Alignment(horizontal="left",   vertical="center")
    al_droite = Alignment(horizontal="right",  vertical="center")
    al_indent = Alignment(horizontal="left",   vertical="center", indent=2)

    # Titre avec l'année
    ws.merge_cells("A1:D1")
    titre_cell = ws.cell(
        row=1, column=1,
        value=f"MODÈLE DE DONNÉES DE SALAIRE DES FONCTIONNAIRES — {annee_label}"
    )
    titre_cell.font      = font_titre
    titre_cell.fill      = fill_titre
    titre_cell.alignment = al_centre
    ws.row_dimensions[1].height = 28

    # En-têtes colonnes
    ws.cell(row=2, column=1, value="salaires en FCFA").font = Font(size=10)
    for col_i, label in enumerate(
        ["salaire brut minimum", "salaire brut moyen", "salaire brut maximum"],
        start=2
    ):
        c = ws.cell(row=2, column=col_i, value=label)
        c.font      = font_entete
        c.fill      = fill_entete
        c.alignment = al_centre
    ws.row_dimensions[2].height = 22

    def get_val(key: str, col: str):
        if key == "_GLOBAL_":
            return global_brut.get(col)
        rows = stats_bareme_detail.filter(pl.col("bareme_detail") == key)
        if len(rows) == 0:
            return None
        return rows.row(0, named=True).get(col)

    row_cur  = 3
    sous_idx = 0
    for libelle, bareme_key in ORDRE_MODELE_ANSTAT:
        est_global   = bareme_key == "_GLOBAL_"
        est_titre    = bareme_key is None
        est_sous_gpe = not est_global and not est_titre

        if est_global:
            for col_i, val in enumerate([
                "TOUS SECTEURS CONFONDUS",
                get_val("_GLOBAL_", "salaire_brut_min"),
                get_val("_GLOBAL_", "salaire_brut_moyen"),
                get_val("_GLOBAL_", "salaire_brut_max"),
            ], start=1):
                c = ws.cell(row=row_cur, column=col_i, value=val)
                c.font      = font_global
                c.fill      = fill_global
                c.alignment = al_gauche if col_i == 1 else al_centre
                if col_i > 1 and isinstance(val, (int, float)):
                    c.number_format = "#,##0"
            ws.row_dimensions[row_cur].height = 22

        elif est_titre:
            ws.merge_cells(
                start_row=row_cur, start_column=1,
                end_row=row_cur,   end_column=4
            )
            c = ws.cell(row=row_cur, column=1, value=libelle)
            c.font      = font_secteur
            c.fill      = fill_secteur
            c.alignment = al_gauche
            ws.row_dimensions[row_cur].height = 20

        else:
            bg = fill_alt[sous_idx % 2]
            sous_idx += 1
            c1 = ws.cell(row=row_cur, column=1, value=libelle)
            c1.font      = font_sous
            c1.fill      = bg
            c1.alignment = al_indent
            for col_i, stat_key in enumerate(
                ["salaire_brut_min", "salaire_brut_moyen", "salaire_brut_max"],
                start=2
            ):
                val = get_val(bareme_key, stat_key)
                c = ws.cell(row=row_cur, column=col_i, value=val)
                c.font          = font_montant
                c.fill          = bg
                c.alignment     = al_droite
                c.number_format = "#,##0"
            ws.row_dimensions[row_cur].height = 18

        row_cur += 1

    # Largeurs colonnes
    ws.column_dimensions["A"].width = 55
    for col_letter in ["B", "C", "D"]:
        ws.column_dimensions[col_letter].width = 22
    ws.freeze_panes = "A3"
    print(f"   ✓ {sheet_name} ({row_cur - 3} lignes)")


# ============================================================
# 1. CHARGEMENT
# ============================================================

print("1. Chargement des fichiers Gold panel...")

pages_panel = s3.get_paginator("list_objects_v2").paginate(
    Bucket=BUCKET_GOLD, Prefix=PREFIX_PANEL
)
cles_panel = sorted([
    obj["Key"]
    for page in pages_panel
    for obj in page.get("Contents", [])
    if re.search(r"panel_20\d{2}\.parquet$", obj["Key"])
])

print(f"   Fichiers panel trouvés : {len(cles_panel)}")

frames = []
for cle in cles_panel:
    print(f"   Lecture {os.path.basename(cle)}...", end=" ", flush=True)
    df_part = lire_parquet_s3(BUCKET_GOLD, cle)
    frames.append(df_part)
    print(f"✓ {len(df_part):,} lignes")

base = pl.concat(frames, how="diagonal")
del frames
gc.collect()
print(f"   ✓ Total : {len(base):,} lignes × {len(base.columns)} colonnes\n")

# ============================================================
# 2. VÉRIFICATION GRADE ET GRADE_SOURCE
# ============================================================

print("2. Vérification des colonnes GRADE et GRADE_SOURCE...")

if "GRADE" not in base.columns or "GRADE_SOURCE" not in base.columns:
    raise RuntimeError(
        "Colonnes GRADE et/ou GRADE_SOURCE absentes du Gold.\n"
        "Exécutez d'abord 06b_imputer_grades.py avant ce script."
    )

n_total = len(base)
for source in ["OBSERVE", "IMPUTE", "NF_NON_IMPUTABLE"]:
    n = int((base["GRADE_SOURCE"] == source).sum())
    print(f"   {source:<20} : {n:,}  ({100*n/n_total:.1f}%)")
print()

# ============================================================
# 3. EXTRACTION ANNÉE / MOIS depuis mois_annee (format MMYYYY)
# ============================================================

print("3. Extraction année/mois depuis mois_annee...")

# mois_annee est au format MMYYYY (ex. "012015")
# mois  = slice(0, 2)  →  "01"
# annee = slice(2, 4)  →  "2015" (4 chiffres à partir de la position 2)
base = base.with_columns([
    pl.col("mois_annee").str.slice(2, 4).cast(pl.Int32, strict=False).alias("ANNEE"),
    pl.col("mois_annee").str.slice(0, 2).cast(pl.Int32, strict=False).alias("MOIS_NUM"),
])

annees_dispo = sorted(base["ANNEE"].drop_nulls().unique().to_list())
print(f"   Années disponibles : {annees_dispo}")

ANNEES_ANALYSE = [2024, 2025] if PERIODE_RECENTE else [a for a in annees_dispo if a >= 2015]
print(f"   Période retenue    : {ANNEES_ANALYSE}")

base = base.filter(pl.col("ANNEE").is_in(ANNEES_ANALYSE))
print(f"   Filtrage           : {len(base):,} lignes\n")

if len(base) == 0:
    raise ValueError(f"Aucune donnée pour les années {ANNEES_ANALYSE}")

base = base.with_columns(
    pl.col("MOIS_NUM")
    .map_elements(lambda m: MOIS_FR.get(m, str(m) if m is not None else ""), return_dtype=pl.Utf8)
    .alias("MOIS")
)

# ============================================================
# 4. CALCUL DU SALAIRE BRUT (composantes élémentaires)
# ============================================================

print("4. Calcul du salaire brut depuis les composantes...")

composantes_presentes = [c for c in COMPOSANTES_SALAIRE if c in base.columns]
print(f"   Composantes trouvées : {composantes_presentes}")

if composantes_presentes:
    base = base.with_columns([
        pl.col(c).cast(pl.Float64, strict=False) for c in composantes_presentes
    ])
    base = base.with_columns(
        pl.sum_horizontal([pl.col(c) for c in composantes_presentes])
        .alias("SALAIRE_BRUT")
    )
else:
    base = base.with_columns(
        pl.col("montant_brut").cast(pl.Float64, strict=False).alias("SALAIRE_BRUT")
        if "montant_brut" in base.columns
        else pl.lit(0.0).alias("SALAIRE_BRUT")
    )

n_pos = int((base["SALAIRE_BRUT"] > 0).sum())
print(f"   Salaires > 0 : {n_pos:,} ({100*n_pos/len(base):.1f}%)\n")

# ============================================================
# 5. STANDARDISATION DU SEXE
# ============================================================

print("5. Standardisation sexe...")

if "sexe" in base.columns:
    base = base.with_columns(
        pl.col("sexe").cast(pl.Utf8)
        .str.to_uppercase()
        .str.strip_chars()
        .map_elements(
            lambda s: (
                "Homme" if s in {"MASCULIN", "M", "H", "HOMME", "MALE", "1"} else
                "Femme" if s in {"FEMININ", "F", "FEMME", "FEMALE", "2",
                                 "FÉMININ", "FEMININE"} else
                "Inconnu"
            ),
            return_dtype=pl.Utf8
        )
        .alias("SEXE_STD")
    )
    n_inconnu = int((base["SEXE_STD"] == "Inconnu").sum())
    if n_inconnu > 0:
        base = base.filter(pl.col("SEXE_STD") != "Inconnu")
        print(f"   Exclusion sexe inconnu : {n_inconnu:,}")
    for sexe_val in ["Homme", "Femme"]:
        n = int((base["SEXE_STD"] == sexe_val).sum())
        print(f"   {sexe_val} : {n:,}")
else:
    base = base.with_columns(pl.lit("Non renseigné").alias("SEXE_STD"))
print()

# ============================================================
# 6. DÉTECTION DES MULTI-POSTES
# ============================================================

print("6. Détection multi-postes (par matricule)...")

if "matricule" in base.columns:
    nb_postes = (
        base.group_by(["mois_annee", "matricule"])
        .agg(pl.len().alias("NB_POSTES"))
    )
    base = base.join(nb_postes, on=["mois_annee", "matricule"], how="left")
    total_multi  = int(
        base.filter(pl.col("NB_POSTES") > 1)
        .select(["mois_annee", "matricule"]).unique().select(pl.len()).item()
    )
    total_agents = int(
        base.select(["mois_annee", "matricule"]).unique().select(pl.len()).item()
    )
    print(f"   Multi-postes : {total_multi:,} / {total_agents:,} "
          f"({100*total_multi/total_agents:.1f}%)\n")
else:
    base = base.with_columns(pl.lit(1).alias("NB_POSTES"))
    print("   Colonne matricule absente, NB_POSTES = 1\n")

# ============================================================
# 7. FILTRAGE SUR GRADES VALIDES
# ============================================================
# On filtre sur GRADE valide (issu de 06b, OBSERVE ou IMPUTE).
# Les NF_NON_IMPUTABLE sont exclus des calculs d'indicateurs.

print("7. Filtrage grades valides avant agrégation...")

if INCLURE_ZEROS:
    base_gv = base.filter(pl.col("GRADE").is_in(GRADES_VALIDES))
else:
    base_gv = base.filter(
        pl.col("GRADE").is_in(GRADES_VALIDES) & (pl.col("SALAIRE_BRUT") > 0)
    )

n_gv = len(base_gv)
for source in ["OBSERVE", "IMPUTE"]:
    n = int((base_gv["GRADE_SOURCE"] == source).sum())
    print(f"   {source:<10} : {n:,}  ({100*n/n_gv:.1f}%)")
print(f"   Total retenu : {n_gv:,} ({100*n_gv/len(base):.1f}%)\n")

# ============================================================
# 7b. FUZZY JOIN EMPLOI → BARÈME
# ============================================================

print("7b. Fuzzy join EMPLOI → BARÈME (via table de correspondance)...")

TABLE_BAREME = charger_table_bareme()

CANDIDATES_EMPLOI = ["emploi", "CODE_EMPLOI", "EMPLOI", "corps", "CORPS"]
col_emploi_panel = next(
    (c for c in CANDIDATES_EMPLOI if c in base_gv.columns), None
)

if TABLE_BAREME is not None and col_emploi_panel is not None:
    base_gv = fuzzy_join_bareme(base_gv, TABLE_BAREME, col_emploi_panel)
    col_grade_bareme = "GRADE" if "GRADE" in base_gv.columns else "grade_ref"
    if col_grade_bareme in base_gv.columns:
        base_gv = ajouter_bareme_detail(base_gv, col_grade_bareme)
    else:
        base_gv = base_gv.with_columns(pl.col("bareme").alias("bareme_detail"))
    BAREME_DISPONIBLE = True
else:
    if TABLE_BAREME is None:
        print("  ⚠️  Table barème absente — feuilles barème non produites")
    else:
        print(f"  ⚠️  Colonne emploi absente dans le panel — feuilles barème non produites")
        print(f"       Colonnes disponibles : {base_gv.columns}")
    BAREME_DISPONIBLE = False

print()

# ============================================================
# 8. AGRÉGATION → REVENU TOTAL
# ============================================================

agg_col   = "cle_unique" if MODE_SALAIRE_BRUT else "matricule"
agg_label = agg_col
print(f"8. Agrégation par {agg_label} → REVENU_TOTAL...")

if agg_col not in base_gv.columns:
    agg_col = "matricule"

base_gv = base_gv.with_columns(
    pl.col("GRADE").map_elements(grade_order_value, return_dtype=pl.Int32)
    .alias("GRADE_ORDER")
)

cols_agg = ["ANNEE", "MOIS_NUM", "MOIS", agg_col]

base_agrege = (
    base_gv
    .group_by(cols_agg)
    .agg([
        pl.col("SALAIRE_BRUT").sum().alias("REVENU_TOTAL"),
        pl.col("NB_POSTES").first().alias("NB_POSTES"),
        pl.col("GRADE").sort_by("GRADE_ORDER", descending=True).first().alias("GRADE_PRINCIPAL"),
        pl.col("GRADE_SOURCE").sort_by("GRADE_ORDER", descending=True).first().alias("GRADE_SOURCE_PRINCIPAL"),
        pl.col("SEXE_STD").first().alias("SEXE_STD"),
        pl.col("Code_CITP").first().alias("Code_CITP")
          if "Code_CITP" in base_gv.columns
          else pl.lit(None).cast(pl.Utf8).alias("Code_CITP"),
        pl.col("Metier_CITP").first().alias("Metier_CITP")
          if "Metier_CITP" in base_gv.columns
          else pl.lit(None).cast(pl.Utf8).alias("Metier_CITP"),
        pl.col("bareme").first().alias("bareme")
          if "bareme" in base_gv.columns
          else pl.lit(None).cast(pl.Utf8).alias("bareme"),
        pl.col("bareme_detail").first().alias("bareme_detail")
          if "bareme_detail" in base_gv.columns
          else pl.lit(None).cast(pl.Utf8).alias("bareme_detail"),
    ])
)

base_gv = base_gv.drop("GRADE_ORDER")

if not INCLURE_ZEROS:
    base_agrege = base_agrege.filter(pl.col("REVENU_TOTAL") > 0)
base_agrege = base_agrege.filter(pl.col("REVENU_TOTAL").is_not_null())

print(f"   {len(base_agrege):,} agents agrégés\n")

# ============================================================
# 8b. WINSORISATION DU REVENU TOTAL
# ============================================================

value_col_indic = "REVENU_TOTAL"

if SEUIL_WINS is not None:
    wins_full     = [c for c in ["ANNEE", "MOIS_NUM", "GRADE_PRINCIPAL", "Code_CITP"]
                     if c in base_agrege.columns]
    wins_fallback = [c for c in ["ANNEE", "MOIS_NUM", "GRADE_PRINCIPAL"]
                     if c in base_agrege.columns]
    print(f"8b. Winsorisation REVENU_TOTAL "
          f"(P{int(SEUIL_WINS*100)}/P{int((1-SEUIL_WINS)*100)}) "
          f"— seuil groupe : {MIN_OBS_WINS} obs...")

    base_agrege = base_agrege.with_columns([
        pl.len().over(wins_full).alias("_n_citp"),
        pl.col("REVENU_TOTAL")
          .quantile(SEUIL_WINS, interpolation="nearest")
          .over(wins_full).alias("_pb_citp"),
        pl.col("REVENU_TOTAL")
          .quantile(1 - SEUIL_WINS, interpolation="nearest")
          .over(wins_full).alias("_ph_citp"),
        pl.col("REVENU_TOTAL")
          .quantile(SEUIL_WINS, interpolation="nearest")
          .over(wins_fallback).alias("_pb_grade"),
        pl.col("REVENU_TOTAL")
          .quantile(1 - SEUIL_WINS, interpolation="nearest")
          .over(wins_fallback).alias("_ph_grade"),
    ])

    n_obs_citp  = int((base_agrege["_n_citp"] >= MIN_OBS_WINS).sum())
    n_obs_grade = len(base_agrege) - n_obs_citp

    base_agrege = base_agrege.with_columns(
        pl.when(pl.col("_n_citp") >= MIN_OBS_WINS)
        .then(pl.col("REVENU_TOTAL").clip(pl.col("_pb_citp"), pl.col("_ph_citp")))
        .otherwise(pl.col("REVENU_TOTAL").clip(pl.col("_pb_grade"), pl.col("_ph_grade")))
        .alias("REVENU_TOTAL_W")
    ).drop(["_n_citp", "_pb_citp", "_ph_citp", "_pb_grade", "_ph_grade"])

    n_modif = int((base_agrege["REVENU_TOTAL_W"] != base_agrege["REVENU_TOTAL"]).sum())
    print(f"   Groupe mois × grade × CITP (≥{MIN_OBS_WINS} obs) : {n_obs_citp:,} obs")
    print(f"   Repli  mois × grade         (<{MIN_OBS_WINS} obs) : {n_obs_grade:,} obs")
    print(f"   Observations écrêtées : {n_modif:,}  ({100*n_modif/len(base_agrege):.2f}%)")
    print(f"   Gold inchangé — winsorisation en mémoire uniquement\n")
    value_col_indic = "REVENU_TOTAL_W"

# ============================================================
# 9. INDICATEURS SUR LE REVENU TOTAL
# ============================================================

print("9. Calcul indicateurs REVENU TOTAL...")

indic_grade = compute_indicators(
    base_agrege, ["ANNEE", "MOIS_NUM", "MOIS", "GRADE_PRINCIPAL"], value_col_indic
)
indic_grade_sexe = compute_indicators(
    base_agrege, ["ANNEE", "MOIS_NUM", "MOIS", "SEXE_STD", "GRADE_PRINCIPAL"], value_col_indic
)

if "Code_CITP" in base_agrege.columns:
    indic_citp = compute_indicators(
        base_agrege.filter(
            pl.col("Code_CITP").is_not_null() & (pl.col("Code_CITP") != "Non renseigné")
        ),
        ["ANNEE", "MOIS_NUM", "MOIS", "Code_CITP", "Metier_CITP"],
        value_col_indic,
    )
else:
    indic_citp = pl.DataFrame()

# Ventilations barème
indic_bareme       = pl.DataFrame()
indic_bareme_grade = pl.DataFrame()
indic_bareme_sexe  = pl.DataFrame()

if BAREME_DISPONIBLE:
    base_b = base_agrege.filter(pl.col("bareme").is_not_null())
    indic_bareme = compute_indicators(
        base_b, ["ANNEE", "MOIS_NUM", "MOIS", "bareme"], value_col_indic
    )
    indic_bareme_grade = compute_indicators(
        base_b, ["ANNEE", "MOIS_NUM", "MOIS", "bareme", "GRADE_PRINCIPAL"], value_col_indic
    )
    indic_bareme_sexe = compute_indicators(
        base_b, ["ANNEE", "MOIS_NUM", "MOIS", "bareme", "SEXE_STD"], value_col_indic
    )

total_indic = (len(indic_grade) + len(indic_grade_sexe) + len(indic_citp)
               + len(indic_bareme) + len(indic_bareme_grade) + len(indic_bareme_sexe))
print(f"   Grade            : {len(indic_grade):,} lignes")
print(f"   Grade × Sexe     : {len(indic_grade_sexe):,} lignes")
print(f"   CITP             : {len(indic_citp):,} lignes")
if BAREME_DISPONIBLE:
    print(f"   Barème           : {len(indic_bareme):,} lignes")
    print(f"   Barème × Grade   : {len(indic_bareme_grade):,} lignes")
    print(f"   Barème × Sexe    : {len(indic_bareme_sexe):,} lignes")
print(f"   Total            : {total_indic:,} lignes\n")

# ============================================================
# 10. INDICATEURS SALAIRE BRUT (par ligne, pivot grade)
# ============================================================

print("10. Calcul indicateurs SALAIRE BRUT (pivot grade)...")

base_gv_brut = base_gv
if SEUIL_WINS is not None:
    wins_full_brut     = [c for c in ["ANNEE", "MOIS_NUM", "GRADE", "Code_CITP"]
                          if c in base_gv.columns]
    wins_fallback_brut = [c for c in ["ANNEE", "MOIS_NUM", "GRADE"]
                          if c in base_gv.columns]

    base_gv_brut = base_gv.with_columns([
        pl.len().over(wins_full_brut).alias("_n_citp"),
        pl.col("SALAIRE_BRUT")
          .quantile(SEUIL_WINS, interpolation="nearest")
          .over(wins_full_brut).alias("_pb_citp"),
        pl.col("SALAIRE_BRUT")
          .quantile(1 - SEUIL_WINS, interpolation="nearest")
          .over(wins_full_brut).alias("_ph_citp"),
        pl.col("SALAIRE_BRUT")
          .quantile(SEUIL_WINS, interpolation="nearest")
          .over(wins_fallback_brut).alias("_pb_grade"),
        pl.col("SALAIRE_BRUT")
          .quantile(1 - SEUIL_WINS, interpolation="nearest")
          .over(wins_fallback_brut).alias("_ph_grade"),
    ])
    base_gv_brut = base_gv_brut.with_columns(
        pl.when(pl.col("_n_citp") >= MIN_OBS_WINS)
        .then(pl.col("SALAIRE_BRUT").clip(pl.col("_pb_citp"), pl.col("_ph_citp")))
        .otherwise(pl.col("SALAIRE_BRUT").clip(pl.col("_pb_grade"), pl.col("_ph_grade")))
        .alias("SALAIRE_BRUT")
    ).drop(["_n_citp", "_pb_citp", "_ph_citp", "_pb_grade", "_ph_grade"])

    n_modif_brut = int((base_gv_brut["SALAIRE_BRUT"] != base_gv["SALAIRE_BRUT"]).sum())
    print(f"   Winsorisation SALAIRE_BRUT (mois × grade × CITP / repli grade) : "
          f"{n_modif_brut:,} obs écrêtées")

brut_moyen = (
    base_gv_brut
    .group_by(["ANNEE", "MOIS_NUM", "MOIS", "GRADE"])
    .agg(pl.col("SALAIRE_BRUT").mean().round(0).alias("Salaire_brut_moyen"))
    .sort(["ANNEE", "MOIS_NUM"])
)
brut_median = (
    base_gv_brut
    .group_by(["ANNEE", "MOIS_NUM", "MOIS", "GRADE"])
    .agg(pl.col("SALAIRE_BRUT").median().round(0).alias("Salaire_brut_median"))
    .sort(["ANNEE", "MOIS_NUM"])
)

brut_moyen_wide = brut_moyen.pivot(
    values="Salaire_brut_moyen",
    index=["ANNEE", "MOIS_NUM", "MOIS"],
    on="GRADE"
).sort(["ANNEE", "MOIS_NUM"])

brut_median_wide = brut_median.pivot(
    values="Salaire_brut_median",
    index=["ANNEE", "MOIS_NUM", "MOIS"],
    on="GRADE"
).sort(["ANNEE", "MOIS_NUM"])

print(f"   Brut moyen  : {len(brut_moyen_wide)} lignes × {len(brut_moyen_wide.columns)} cols")
print(f"   Brut médian : {len(brut_median_wide)} lignes × {len(brut_median_wide.columns)} cols\n")

# ============================================================
# 11. MULTI-POSTES
# ============================================================

print("11. Création feuilles multi-postes...")

multi_detail = (
    base_agrege
    .group_by(["ANNEE", "MOIS_NUM", "MOIS", "GRADE_PRINCIPAL", "NB_POSTES"])
    .agg([
        pl.len().alias("Effectif"),
        pl.col("REVENU_TOTAL").mean().round(0).alias("Revenu_moyen"),
    ])
    .sort(["ANNEE", "MOIS_NUM", "GRADE_PRINCIPAL", "NB_POSTES"])
)

multi_detail = multi_detail.with_columns(
    pl.when(pl.col("NB_POSTES") >= 5)
    .then(pl.lit("5+"))
    .otherwise(pl.col("NB_POSTES").cast(pl.Utf8))
    .alias("NB_POSTES_LABEL")
)

multi_effectif = multi_detail.pivot(
    values="Effectif",
    index=["ANNEE", "MOIS_NUM", "MOIS", "GRADE_PRINCIPAL"],
    on="NB_POSTES_LABEL",
    aggregate_function="sum",
).sort(["ANNEE", "MOIS_NUM", "GRADE_PRINCIPAL"])

for c in ["1", "2", "3", "4", "5+"]:
    if c in multi_effectif.columns:
        multi_effectif = multi_effectif.with_columns(pl.col(c).fill_null(0))

distrib_postes = (
    base_agrege
    .group_by("NB_POSTES")
    .agg([
        pl.len().alias("Effectif"),
        (pl.len() * 100 / len(base_agrege)).round(2).alias("Pct"),
        pl.col("REVENU_TOTAL").mean().round(0).alias("Revenu_moyen"),
    ])
    .sort("NB_POSTES")
)

print(f"   Multi-postes détail  : {len(multi_effectif)} lignes")
print(f"   Distribution globale : {len(distrib_postes)} lignes\n")

# ============================================================
# 12. PRÉ-CALCUL DONNÉES MODELE_ANSTAT PAR ANNÉE
# ============================================================
# On prépare un dict {annee: (global_brut, stats_bareme_detail)}
# pour éviter de refiltrer base_gv_b plusieurs fois.

anstat_data_par_annee: dict[int, tuple[dict, pl.DataFrame]] = {}

if BAREME_DISPONIBLE:
    print("12. Pré-calcul données MODELE_ANSTAT par année...")
    for annee in [2024, 2025]:
        if annee not in ANNEES_ANALYSE:
            print(f"   ⚠️  Année {annee} absente de la période — ignorée")
            continue

        base_gv_b_annee = base_gv.filter(
            (pl.col("ANNEE") == annee)
            & pl.col("bareme_detail").is_not_null()
            & pl.col("SALAIRE_BRUT").is_not_null()
            & (pl.col("SALAIRE_BRUT") > 0)
        )

        if len(base_gv_b_annee) == 0:
            print(f"   ⚠️  Aucune donnée barème pour {annee} — feuille ignorée")
            continue

        global_brut_annee = base_gv_b_annee.select([
            pl.col("SALAIRE_BRUT").min().round(0).alias("salaire_brut_min"),
            pl.col("SALAIRE_BRUT").mean().round(0).alias("salaire_brut_moyen"),
            pl.col("SALAIRE_BRUT").max().round(0).alias("salaire_brut_max"),
            pl.col("SALAIRE_BRUT").median().round(0).alias("salaire_brut_mediane"),
        ]).row(0, named=True)

        stats_annee = (
            base_gv_b_annee.group_by("bareme_detail")
            .agg([
                pl.col("SALAIRE_BRUT").min().round(0).alias("salaire_brut_min"),
                pl.col("SALAIRE_BRUT").mean().round(0).alias("salaire_brut_moyen"),
                pl.col("SALAIRE_BRUT").max().round(0).alias("salaire_brut_max"),
                pl.col("SALAIRE_BRUT").median().round(0).alias("salaire_brut_mediane"),
                pl.len().alias("nb_obs"),
            ])
            .sort("bareme_detail")
        )

        anstat_data_par_annee[annee] = (global_brut_annee, stats_annee)
        print(f"   ✓ {annee} : {len(stats_annee)} barèmes, "
              f"{len(base_gv_b_annee):,} observations")
    print()

# ============================================================
# 13. EXPORT EXCEL
# ============================================================

print("13. Export Excel...")

def appliquer_style_entete(ws, n_cols: int) -> None:
    fill  = PatternFill("solid", fgColor="4F81BD")
    font  = Font(color="FFFFFF", bold=True, size=11)
    align = Alignment(horizontal="center", vertical="center")
    for col in range(1, n_cols + 1):
        cell = ws.cell(row=1, column=col)
        cell.fill      = fill
        cell.font      = font
        cell.alignment = align


def ecrire_feuille(wb: Workbook, nom: str, df: pl.DataFrame) -> None:
    if len(df) == 0:
        return
    ws = wb.create_sheet(title=nom)
    for j, col in enumerate(df.columns, start=1):
        ws.cell(row=1, column=j, value=col)
    for i, row in enumerate(df.iter_rows(named=True), start=2):
        for j, col in enumerate(df.columns, start=1):
            ws.cell(row=i, column=j, value=row[col])
    appliquer_style_entete(ws, len(df.columns))
    ws.freeze_panes = "A2"
    print(f"   ✓ {nom} ({len(df):,} lignes)")


wb = Workbook()
wb.remove(wb.active)

ecrire_feuille(wb, "REVENU_Grade_Detail",      indic_grade)
ecrire_feuille(wb, "REVENU_Grade_Sexe",         indic_grade_sexe)
if len(indic_citp) > 0:
    ecrire_feuille(wb, "REVENU_CITP_Detail",    indic_citp)
ecrire_feuille(wb, "SALAIRE_BRUT_Moyen",        brut_moyen_wide)
ecrire_feuille(wb, "SALAIRE_BRUT_Median",       brut_median_wide)
ecrire_feuille(wb, "MULTI_Par_Grade_NbPostes",  multi_effectif)
ecrire_feuille(wb, "MULTI_Distribution",        distrib_postes)

# --- Feuilles barème (si disponibles) ---
if BAREME_DISPONIBLE:
    ecrire_feuille(wb, "REVENU_Bareme_Detail",  indic_bareme)
    ecrire_feuille(wb, "REVENU_Bareme_Grade",   indic_bareme_grade)
    ecrire_feuille(wb, "REVENU_Bareme_Sexe",    indic_bareme_sexe)

    # --- Feuilles MODELE_ANSTAT_2024 et MODELE_ANSTAT_2025 ---
    # Une feuille par année, construite depuis les données filtrées
    # calculées à l'étape 12.
    for annee in [2024, 2025]:
        if annee not in anstat_data_par_annee:
            print(f"   ⚠️  MODELE_ANSTAT_{annee} ignorée (données absentes)")
            continue
        global_brut_annee, stats_annee = anstat_data_par_annee[annee]
        construire_feuille_anstat(
            wb=wb,
            sheet_name=f"MODELE_ANSTAT_{annee}",
            annee_label=str(annee),
            global_brut=global_brut_annee,
            stats_bareme_detail=stats_annee,
        )

buf_xlsx = io.BytesIO()
wb.save(buf_xlsx)
buf_xlsx.seek(0)
xlsx_bytes = buf_xlsx.getvalue()

s3.put_object(
    Bucket=BUCKET_STAGING,
    Key=OUTPUT_KEY,
    Body=xlsx_bytes,
    ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
)

taille_kb = len(xlsx_bytes) / 1024
print(f"\n✅ Fichier créé : s3://{BUCKET_STAGING}/{OUTPUT_KEY}")
print(f"   Taille   : {taille_kb:.1f} KB")
print(f"   Feuilles : {len(wb.sheetnames)}\n")

print("=" * 70)
print("✓ CALCUL INDICATEURS TERMINÉ")
print("=" * 70)
print()
print("Prochaine étape : exécuter 08_creation_fichier_excel_avec_sommaire.py")

# ============================================================
# 14. SAUVEGARDE DE LA BASE TRAITÉE
# ============================================================
print("14. Détection agents hors plafond barème...")

# Vérification que REVENU_TOTAL_W existe bien (winsorisation activée)
if "REVENU_TOTAL_W" not in base_agrege.columns:
    raise RuntimeError(
        "La colonne REVENU_TOTAL_W est absente — "
        "vérifiez que SEUIL_WINS est bien défini (actuellement None)."
    )

# Correspondance Excel → valeurs du parquet
MAPPING_SECTEUR = {
    "Barême Général"         : "Barême Général",                          # identique
    "Cadre-Sup-Santé"        : "Cadre-Sup-Santé Generaliste & Specialiste",
    "CORPS DIPL"             : "CORPS DIPL",                             # identique
    "CORPS PREF"             : "CORPS PREF",                             # identique
    "ENS SEC & PRIM"         : "ENS SEC & PRIM",                         # identique
    "ENS SUP"                : "ENS SUP",                                # identique
    "GREFFE"                 : "GREFFE",                                  # identique
    "Informaticien"          : "Informaticien",   # absent du parquet
    "MAGISTRAT "             : "MAGISTRAT ",                             # identique (espace inclus)
    "para militaire"         : "police",
    "SECRETAIRE DIRECTION"   : "SECRETAIRE DIRECTION",   # absent du parquet 
    "Tech-Santé"             : "Tech-Santé",                             # identique
}

try:
    bucket_name = "staging"
    object_key  = "panel_admin/exports_gold/stats_secteur_grade_Solde.xlsx"
    obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
    buf = io.BytesIO(obj['Body'].read())
    buf.seek(0)

    table_max_brut = pl.from_pandas(
        pd.read_excel(buf)[["SECTEUR", "GRADE", "max_brut"]]
    )

    # Application du mapping : renommer les secteurs Excel → noms parquet
    table_max_brut = (
        table_max_brut
        .with_columns(
            pl.col("SECTEUR")
            .map_elements(lambda s: MAPPING_SECTEUR.get(s, s), return_dtype=pl.Utf8)
            .alias("SECTEUR")
        )
        # Supprimer les lignes sans correspondance dans le parquet (None)
        .filter(pl.col("SECTEUR").is_not_null())
        .rename({"GRADE": "GRADE_REF", "SECTEUR": "BAREME_REF"})
    )

    print(f"  ✓ Table max_brut chargée : {len(table_max_brut)} lignes")
    print(f"  ✓ Secteurs après mapping : {sorted(table_max_brut['BAREME_REF'].unique().to_list())}")

except Exception as e:
    print(f"  ⚠️  Table max_brut non disponible ({e}) — étape 11 ignorée")
    table_max_brut = None

if table_max_brut is not None:
    base_agrege = (
        base_agrege
        .join(
            table_max_brut,
            left_on  = ["GRADE_PRINCIPAL", "bareme"],
            right_on = ["GRADE_REF", "BAREME_REF"],
            how      = "left",
        )
.with_columns(
            pl.when(pl.col("bareme").is_null() | (pl.col("bareme") == ""))
            .then(pl.lit("Indéterminé"))
            .when(
                pl.col("max_brut").is_not_null() &
                (pl.col("REVENU_TOTAL_W") > pl.col("max_brut"))
            )
            .then(pl.lit("Au dessus du max"))
            .otherwise(pl.lit("En dessous du max"))
            .alias("STATUT")
        )
    )

    n_total       = len(base_agrege)
    n_dessus      = int((base_agrege["STATUT"] == "Au dessus du max").sum())
    n_dessous     = int((base_agrege["STATUT"] == "En dessous du max").sum())
    n_indetermine = int((base_agrege["STATUT"] == "Indéterminé").sum())

    print(f"   Au dessus du max  : {n_dessus:,}      ({100 * n_dessus      / n_total:.2f}%)")
    print(f"   En dessous du max : {n_dessous:,}     ({100 * n_dessous     / n_total:.2f}%)")
    print(f"   Indéterminé       : {n_indetermine:,} ({100 * n_indetermine / n_total:.2f}%)")
    print(f"   Total             : {n_total:,}")
    print(f"   [Comparaison effectuée sur REVENU_TOTAL_W — revenu winsorisé P1/P99]\n")

else:
    base_agrege = base_agrege.with_columns(
        pl.lit(False).alias("HORS_PLAFOND"),
        pl.lit(None).cast(pl.Float64).alias("max_brut"),
    )
    print("  ⚠️  HORS_PLAFOND mis à False pour tous les agents\n")

# ============================================================
# 15. SAUVEGARDE FINALE DE LA BASE TRAITÉE
# ============================================================
# ← ICI, au même niveau que les blocs if/else, pas à l'intérieur
print("12. Sauvegarde finale de la base traitée...")

try:
    buf_save = io.BytesIO()
    base_agrege.write_parquet(buf_save)
    buf_save.seek(0)
    key_save = f"{PREFIX_EXPORTS}/base_agrege_{_mode_label}_{_periode_label}_{_zeros_label}.parquet"
    s3.upload_fileobj(buf_save, BUCKET_STAGING, key_save)
    print(f"  ✓ base_agrege sauvegardée : {len(base_agrege):,} lignes → {key_save}")
    print(f"  ✓ Colonnes sauvegardées   : {base_agrege.columns}")

except Exception as e:
    print(f"  ⚠️  Sauvegarde échouée ({e})")

print()
# -----------------------------
# Filtrer agents au-dessus du plafond
# -----------------------------
base_ecart = agents_detail.filter(pl.col("STATUT") == "Au dessus du max")
print(f"Agents au-dessus du plafond : {base_ecart.height:,}")

# -----------------------------
# Calcul de l'écart au plafond
# -----------------------------
agents_detail = agents_detail.with_columns([
    pl.when(pl.col("STATUT") == "Au dessus du max")
      .then(pl.col("REVENU_TOTAL_W") - pl.col("max_brut"))
      .otherwise(None)
      .alias("ECART_MAX"),

    pl.when((pl.col("STATUT") == "Au dessus du max") & (pl.col("max_brut") > 0))
      .then((pl.col("REVENU_TOTAL_W") - pl.col("max_brut")) / pl.col("max_brut") * 100)
      .otherwise(None)
      .alias("ECART_MAX_PCT")
])

# -----------------------------
# Sauvegarder la base mise à jour vers S3 si besoin
# -----------------------------
output_buffer = io.BytesIO()
agents_detail.write_parquet(output_buffer)
output_buffer.seek(0)

s3_client.put_object(
    Bucket=bucket_name,
    Key="panel_admin/exports_gold/agents_au_dessus_max.parquet",
    Body=output_buffer
)
# Ton DataFrame filtré
df_sup = df[df['STATUT'] == "Au dessus du max"].copy()

# Buffer mémoire
buffer = io.BytesIO()

# Écriture parquet dans le buffer
df_sup.to_parquet(buffer, index=False)
buffer.seek(0)

# Upload sur MinIO
s3_client.put_object(
    Bucket="staging",
    Key="panel_admin/exports_gold/agents_au_dessus_max.parquet",
    Body=buffer.getvalue()
)

print("✅ Fichier agents_au_dessus_max.parquet sauvegardé sur le serveur MinIO !")



INDICATEURS SALAIRES — Mode : Salaire brut (cle_unique) | Période : 2024-2025

1. Chargement des fichiers Gold panel...
   Fichiers panel trouvés : 11
   Lecture panel_2015.parquet... ✓ 2,264,346 lignes
   Lecture panel_2016.parquet... ✓ 2,183,613 lignes
   Lecture panel_2017.parquet... ✓ 2,460,161 lignes
   Lecture panel_2018.parquet... ✓ 2,578,098 lignes
   Lecture panel_2019.parquet... ✓ 2,662,315 lignes
   Lecture panel_2020.parquet... ✓ 2,822,408 lignes
   Lecture panel_2021.parquet... 

In [ ]:
# Paramètres
import io
import pandas as pd
import boto3
import pandas as pd
import io
import pandas as pd
import unicodedata
import re
import numpy as np
import unicodedata, re
from pathlib import Path
from pandas.api.types import is_string_dtype
from collections import Counter
from rapidfuzz import process, fuzz
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

In [ ]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://192.168.1.230:30137", # service interne K8s
 aws_access_key_id="datalab-team",
 aws_secret_access_key="minio-datalabteam123",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "staging"
object_key = "panel_admin/exports_gold/base_agrege_salaire_brut_2024_2025_avec_zeros.parquet"

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
df = pd.read_parquet(io.BytesIO(obj['Body'].read()))
# Afficher un aperçu
df.head()

,ANNEE,MOIS_NUM,MOIS,cle_unique,REVENU_TOTAL,NB_POSTES,GRADE_PRINCIPAL,GRADE_SOURCE_PRINCIPAL,SEXE_STD,Code_CITP,Metier_CITP,bareme,bareme_detail,REVENU_TOTAL_W,max_brut,STATUT
0,2024,10,Octobre,884335Q38,175465.0,1,C3,OBSERVE,Homme,0210,Sous-officiers des forces armées,police,police,175465.0,436121.892,En dessous du max
1,2024,1,Janvier,391447S22,305889.0,1,C3,OBSERVE,Femme,2341,Enseignants du primaire,ENS SEC & PRIM,ENS SEC & PRIM_PRIM,305889.0,362501.516,En dessous du max
2,2025,3,Mars,849586Z38,252677.0,1,C3,IMPUTE,Homme,0210,Sous-officiers des forces armées,police,police,252677.0,436121.892,En dessous du max
3,2024,8,Août,318657J22,590781.0,1,B3,OBSERVE,Homme,2341,Enseignants du primaire,ENS SEC & PRIM,ENS SEC & PRIM_PRIM,590781.0,557145.456,Au dessus du max
4,2025,8,Août,297281WFC,685903.0,1,A4,OBSERVE,Femme,2149,Professionnels de l'ingénierie non classés ail...,Tech-Santé,Tech-Santé,685903.0,752453.868,En dessous du max


In [ ]:
print ("Les variables de cette base de données sont :", df.columns)
print ("Le nombre d'observations est :", len(df))

Les variables de cette base de données sont : Index(['ANNEE', 'MOIS_NUM', 'MOIS', 'cle_unique', 'REVENU_TOTAL', 'NB_POSTES',
       'GRADE_PRINCIPAL', 'GRADE_SOURCE_PRINCIPAL', 'SEXE_STD', 'Code_CITP',
       'Metier_CITP', 'bareme', 'bareme_detail', 'REVENU_TOTAL_W', 'max_brut',
       'STATUT'],
      dtype='object')
Le nombre d'observations est : 7597798


In [ ]:
df['STATUT'].value_counts()

STATUT
En dessous du max    6719231
Au dessus du max      877626
Indéterminé              941
Name: count, dtype: int64

In [ ]:
df['STATUT'].unique()

array(['En dessous du max', 'Au dessus du max', 'Indéterminé'],
      dtype=object)

In [ ]:
# Effectifs (avec toutes les modalités, y compris NaN)
effectifs = df['STATUT'].value_counts(dropna=False)

# Proportions en %
proportions = round(df['STATUT'].value_counts(dropna=False, normalize=True) * 100, 2)

# Tableau final
tab_statut = pd.DataFrame({
    "Effectif": effectifs,
    "Proportion (%)": proportions
})

tab_statut

,Effectif,Proportion (%)
STATUT,,
En dessous du max,6719231,88.44
Au dessus du max,877626,11.55
Indéterminé,941,0.01


In [ ]:
count_sup = (df['REVENU_TOTAL_W'] > df['max_brut']).sum()
count_sup

np.int64(877626)

In [ ]:
count_inf = (df['REVENU_TOTAL_W'] < df['max_brut']).sum()
count_inf

np.int64(6719231)

In [ ]:
count_equ = (df['REVENU_TOTAL_W'] == df['max_brut']).sum()
count_equ

np.int64(0)

✅ Fichier agents_au_dessus_max.parquet sauvegardé sur le serveur MinIO !


In [ ]:
import boto3
import polars as pl
import io

# -----------------------------
# Connexion S3
# -----------------------------
s3_client = boto3.client(
    "s3",
    endpoint_url="http://192.168.1.230:30137",  # service interne K8s
    aws_access_key_id="datalab-team",
    aws_secret_access_key="minio-datalabteam123",
    verify=False  # SSL auto-signé
)

# -----------------------------
# Paramètres
# -----------------------------
bucket_name = "staging"
object_key = "panel_admin/exports_gold/agents_au_dessus_max.parquet"

# -----------------------------
# Télécharger et charger en Polars
# -----------------------------
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
agents_detail = pl.read_parquet(io.BytesIO(obj['Body'].read()))





,ANNEE,MOIS_NUM,MOIS,cle_unique,REVENU_TOTAL,NB_POSTES,GRADE_PRINCIPAL,GRADE_SOURCE_PRINCIPAL,SEXE_STD,Code_CITP,Metier_CITP,bareme,bareme_detail,REVENU_TOTAL_W,max_brut,STATUT
0,2024,8,Août,318657J22,590781.0,1,B3,OBSERVE,Homme,2341,Enseignants du primaire,ENS SEC & PRIM,ENS SEC & PRIM_PRIM,590781.0,557145.456,Au dessus du max
1,2025,5,Mai,342688J22,447908.0,1,C3,OBSERVE,Homme,2341,Enseignants du primaire,ENS SEC & PRIM,ENS SEC & PRIM_PRIM,447908.0,362501.516,Au dessus du max
2,2024,10,Octobre,393585P17,475986.0,1,B3,OBSERVE,Homme,5413,Gardiens de prison,Barême Général,Barême Général,475986.0,445728.580,Au dessus du max
3,2025,4,Avril,253848R15,796274.0,1,A3,OBSERVE,Homme,2330,Enseignants du secondaire,ENS SEC & PRIM,ENS SEC & PRIM_SEC,796274.0,761186.000,Au dessus du max
4,2024,6,Juin,319669X22,363894.0,1,C3,OBSERVE,Homme,2341,Enseignants du primaire,ENS SEC & PRIM,ENS SEC & PRIM_PRIM,363894.0,362501.516,Au dessus du max


In [ ]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://192.168.1.230:30137", # service interne K8s
 aws_access_key_id="datalab-team",
 aws_secret_access_key="minio-datalabteam123",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "staging"
object_key = "panel_admin/exports_gold/agents_au_dessus_max_updated.parquet"

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
df = pd.read_parquet(io.BytesIO(obj['Body'].read()))
# Afficher un aperçu
df.head()

NoSuchKey: An error occurred (NoSuchKey) when calling the GetObject operation: The specified key does not exist.

: 

In [ ]:
# Afficher les variables de la base de données
print ("Les variables de cette base de données sont :", df.columns)
# Afficher le nombre d'observations
print ("Le nombre d'observations est :", len(df))

Les variables de cette base de données sont : Index(['ANNEE', 'MOIS_NUM', 'MOIS', 'cle_unique', 'REVENU_TOTAL', 'NB_POSTES',
       'GRADE_PRINCIPAL', 'GRADE_SOURCE_PRINCIPAL', 'SEXE_STD', 'Code_CITP',
       'Metier_CITP', 'bareme', 'bareme_detail', 'REVENU_TOTAL_W', 'max_brut',
       'STATUT'],
      dtype='object')
Le nombre d'observations est : 877626


In [ ]:
# ============================================================
# 6. CALCUL DE L'ÉCART AU PLAFOND (simplifié)
# ============================================================
print("6. Calcul de l'écart au plafond...")

import polars as pl

# ============================================================
# 6. CALCUL DE L'ÉCART AU PLAFOND (simplifié Polars)
# ============================================================
print("6. Calcul de l'écart au plafond...")

agents_detail = agents_detail.with_columns([
    # Écart absolu
    pl.when(pl.col("STATUT") == "Au dessus du max")
      .then(pl.col("REVENU_TOTAL_W") - pl.col("max_brut"))
      .otherwise(None)
      .alias("ECART_MAX"),

    # Écart relatif (%)
    pl.when((pl.col("STATUT") == "Au dessus du max") & (pl.col("max_brut") > 0))
      .then((pl.col("REVENU_TOTAL_W") - pl.col("max_brut")) / pl.col("max_brut") * 100)
      .otherwise(None)
      .alias("ECART_MAX_PCT")
])

# Filtrer les agents au-dessus du plafond
base_ecart = agents_detail.filter(pl.col("STATUT") == "Au dessus du max")

# Affichage de l'effectif
print(f"   Agents au-dessus du plafond : {base_ecart.height:,}\n")

6. Calcul de l'écart au plafond...
6. Calcul de l'écart au plafond...


NameError: name 'agents_detail' is not defined

In [ ]:
df['ECART_MAX'].dtype

KeyError: 'ECART_MAX'

In [ ]:
print("Nombre de NaN :", df['ECART_MAX'].isna().sum())
print("Nombre de zéros :", (df['ECART_MAX'] == 0).sum())
print("Nombre de valeurs négatives :", (df['ECART_MAX'] < 0).sum())
print("\nRésumé statistique :")
print(df['ECART_MAX'].describe())

Nombre de NaN : 0
Nombre de zéros : 1607
Nombre de valeurs négatives : 0

Résumé statistique :
count    8.776260e+05
mean     2.724558e+05
std      7.959805e+05
min      0.000000e+00
25%      1.972900e+04
50%      7.380500e+04
75%      1.344450e+05
max      2.730974e+07
Name: ECART_MAX, dtype: float64


In [ ]:
# Effectifs (inclut NaN)
effectifs = df['ECART_MAX'].value_counts(dropna=False)

# Proportions en %
proportions = df['ECART_MAX'].value_counts(dropna=False, normalize=True) * 100

# Tableau initial
tab_ecart_max = pd.DataFrame({
    "Effectif": effectifs,
    "Proportion (%)": proportions
})

# Index auxiliaire pour placer NaN à la fin
tab_ecart_max["index_aux"] = tab_ecart_max.index.to_series().apply(
    lambda x: float('inf') if pd.isna(x) else x
)

# Tri
tab_ecart_max = tab_ecart_max.sort_values("index_aux")

# Calcul cumul AVANT arrondi
tab_ecart_max["Proportion cumulée (%)"] = tab_ecart_max["Proportion (%)"].cumsum()

# Arrondir à la fin
tab_ecart_max[["Proportion (%)", "Proportion cumulée (%)"]] = \
    tab_ecart_max[["Proportion (%)", "Proportion cumulée (%)"]].round(2)

# Supprimer colonne auxiliaire
tab_ecart_max = tab_ecart_max.drop(columns="index_aux")
tab_ecart_max.sort_index().to_excel(
    r"C:\Users\a_coulibaly\Downloads\tabulation_ECART_MAX.xlsx",
    index=True
)

In [ ]:
cluster_stats = (
    df.groupby(['GRADE_PRINCIPAL', 'bareme'])
      .agg(
          MEDIAN_ECART_MAX = ('ECART_MAX', 'median'),
          EFFECTIF = ('ECART_MAX', 'count')
      )
      .reset_index()
)

cluster_stats
cluster_stats.sort_index().to_excel(
    r"C:\Users\a_coulibaly\Downloads\cluster_stats.xlsx",
    index=True
)

In [ ]:
# ===============================
# 1. Effectifs et proportions RAW
# ===============================

# Effectifs
effectifs = df['ECART_MAX_PCT'].value_counts(dropna=False)

# Proportions NON-arrondies (important pour cumul = 100%)
proportions_raw = df['ECART_MAX_PCT'].value_counts(dropna=False, normalize=True) * 100

# ===============================
# 2. Construction tableau final
# ===============================

tab_ecart_pct = pd.DataFrame({
    "Effectif": effectifs,
    "Proportion (%)": proportions_raw.round(2)
})

# Tri correct par valeurs ECART_MAX_PCT
tab_ecart_pct = tab_ecart_pct.sort_index()

# ===================================
# 3. Cumul sur proportions NON arrondies
# ===================================

tab_ecart_pct["Proportion cumulée (%)"] = proportions_raw.sort_index().cumsum().round(2)

# ===============================
# 4. Export Excel
# ===============================

chemin = r"C:\Users\a_coulibaly\Downloads\distribution_ECART_MAX_PCT.xlsx"
tab_ecart_pct.to_excel(chemin, index=True)

print("✅ Exporté avec succès :", chemin)

# ===============================
# 5. Affichage final
# ===============================
tab_ecart_pct

✅ Exporté avec succès : C:\Users\a_coulibaly\Downloads\distribution_ECART_MAX_PCT.xlsx


,Effectif,Proportion (%),Proportion cumulée (%)
ECART_MAX_PCT,,,
0.00,20211,2.30,2.30
0.01,661,0.08,2.38
0.02,63,0.01,2.39
0.03,100,0.01,2.40
0.04,825,0.09,2.49
...,...,...,...
1876.58,1,0.00,100.00
1881.02,9,0.00,100.00
1958.04,3,0.00,100.00


In [ ]:
# --- Sous-base : uniquement les "Au dessus du max"
df_sup = df[df['STATUT'] == "Au dessus du max"].copy()


# Nouvelle variable à imputer
col = "REVENU_TOTAL_W"
new_col = "REVENU_TOTAL_W_IMP"



# ======================================================
# 1) Médiane cluster L1 : GRADE_PRINCIPAL × bareme
# ======================================================
cluster_L1 = (
    df_sup.groupby(['GRADE_PRINCIPAL', 'bareme'])[col]
          .agg(['median', 'count'])
          .reset_index()
          .rename(columns={'median': 'med_L1', 'count': 'n_L1'})
)

# Garder seulement les clusters valides
cluster_L1_valid = cluster_L1[cluster_L1['n_L1'] >= 5][['GRADE_PRINCIPAL', 'bareme', 'med_L1']]


# ======================================================
# 2) Médiane L2 : bareme seul
# ======================================================
cluster_L2 = (
    df_sup.groupby('bareme')[col]
          .median()
          .reset_index()
          .rename(columns={col: 'med_L2'})
)

# ======================================================
# 3) Médiane globale L3
# ======================================================

med_L3 = df_sup[col].median()

# ======================================================
# Imputation finale
# ======================================================
# Fusion successivement des médianes
df_sup = df_sup.merge(cluster_L1_valid, on=['GRADE_PRINCIPAL', 'bareme'], how='left')
df_sup = df_sup.merge(cluster_L2, on='bareme', how='left')

# Imputation avec cascade L1 → L2 → L3
df_sup[new_col] = (
    df_sup[col]
      .fillna(df_sup['med_L1'])
      .fillna(df_sup['med_L2'])
      .fillna(med_L3)
)